# Demo 3 — Responsible AI

**FarmWise, Session 1 · Africa Digital Skills Conference**

Two AI features that *work* is not the same as two features you should *trust*.
We look at:

1. Where the crop data came from — and why that matters for Ghana.
2. Flagging inputs the model never saw during training (out-of-distribution).
3. Prediction confidence, and when to distrust it.
4. Per-crop performance (averages hide weak spots).
5. Hallucination, and why RAG-with-citations helps.
6. An honest list of limitations.

### Setup

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt

from farmwise import config
from farmwise.ml.data import load_crop_dataset, FEATURE_COLUMNS

## 1 — Where did the data come from?

The model is only as trustworthy as its data. We stored the provenance in the
model metadata on purpose.

In [2]:
meta = json.loads(config.MODEL_METADATA_PATH.read_text())
print(meta["dataset_source"])

Kaggle 'Crop Recommendation' dataset (Atharva Ingle). Collection location and methodology are undocumented; widely understood to reflect Indian agronomic conditions — NOT validated for Ghana.


This dataset has **no documented collection location** and is widely understood
to reflect **Indian** conditions. Our workshop is in **Ghana**. So the key question:
*do Ghanaian farms even fall inside the ranges this model learned from?*

## 2 — Compare training ranges with plausible Ghana conditions

Below, the training min/max for each feature vs *rough, illustrative* ranges a
Ghanaian farm might see. (The Ghana numbers are indicative for teaching, not a
soil survey.) We flag any feature whose plausible local range falls **outside**
what the model was trained on.

In [3]:
ranges = meta["feature_ranges"]

# Illustrative only — for discussion, not agronomic reference values.
ghana_plausible = {
    "N": (0, 140), "P": (5, 145), "K": (5, 205),
    "temperature": (22, 40),   # Ghana can exceed the training max in the north
    "humidity": (40, 95),
    "ph": (4.5, 7.5),
    "rainfall": (20, 300),
}

table = []
for f in FEATURE_COLUMNS:
    tlo, thi = ranges[f]["min"], ranges[f]["max"]
    glo, ghi = ghana_plausible[f]
    outside = glo < tlo or ghi > thi
    table.append({
        "feature": f,
        "train_min": round(tlo, 1), "train_max": round(thi, 1),
        "ghana_low": glo, "ghana_high": ghi,
        "ghana exceeds training range?": "YES" if outside else "no",
    })
pd.DataFrame(table)

,feature,train_min,train_max,ghana_low,ghana_high,ghana exceeds training range?
0,N,0.0,140.0,0.0,140.0,no
1,P,5.0,145.0,5.0,145.0,no
2,K,5.0,205.0,5.0,205.0,no
3,temperature,8.8,43.7,22.0,40.0,no
4,humidity,14.3,100.0,40.0,95.0,no
5,ph,3.5,9.9,4.5,7.5,no
6,rainfall,20.2,298.6,20.0,300.0,YES


Where it says **YES**, real Ghanaian inputs could land in territory the model
never saw — predictions there are extrapolation, not evidence.

## 3 — The model flags out-of-distribution inputs

We built this check into `predict_crop`: if any input is outside the training
range, the result is marked untrustworthy with a warning.

In [4]:
from farmwise.ml.predict import predict_crop, CropFeatures

# A hot northern-Ghana-ish reading: temperature above the training max.
risky = CropFeatures(N=80, P=40, K=40, temperature=45, humidity=55, ph=6.2, rainfall=120)
pred = predict_crop(risky)

print("Recommended:", pred.crop, "| confidence:", round(pred.confidence, 3))
print("Out of distribution?", pred.out_of_distribution)
for w in pred.ood_warnings:
    print("  !", w)

Recommended: coffee | confidence: 0.705
Out of distribution? True
  ! temperature=45 is outside the training range [8.82567, 43.6755]


## 4 — Confidence: when to distrust a prediction

The model always returns *a* crop. The top-3 probabilities tell you how sure it
is. A near-tie means "don't bet the season on this."

In [5]:
examples = {
    "typical rice-ish farm": CropFeatures(90, 42, 43, 21, 82, 6.5, 200),
    "ambiguous / in-between": CropFeatures(60, 60, 60, 26, 60, 6.5, 100),
}
for name, feats in examples.items():
    p = predict_crop(feats)
    top3 = [(s.crop, round(s.probability, 2)) for s in p.top3]
    print(f"{name:24s} -> {p.crop:12s} conf={p.confidence:.2f}  top3={top3}")

typical rice-ish farm    -> rice         conf=0.92  top3=[('rice', 0.92), ('jute', 0.09), ('apple', 0.0)]
ambiguous / in-between   -> maize        conf=0.26  top3=[('maize', 0.26), ('papaya', 0.2), ('coffee', 0.18)]


## 5 — Per-crop performance

Overall accuracy is an average. Some crops are predicted far better than others.
Let's retrain to get the full per-class report and show the weakest crops.

In [6]:
from farmwise.ml.train import train_crop_model

df = load_crop_dataset()
result = train_crop_model(df)

report = pd.DataFrame(result.report_dict).T
per_class = report.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
per_class[["precision", "recall", "f1-score"]].sort_values("f1-score").head(6).round(3)

,precision,recall,f1-score
blackgram,1.000,0.95,0.974
rice,1.000,0.95,0.974
maize,0.952,1.00,0.976
jute,0.952,1.00,0.976
coconut,1.000,1.00,1.000
coffee,1.000,1.00,1.000


## 6 — Hallucination, and why RAG with citations helps

A plain LLM will answer almost anything, even with nothing to back it up. The RAG
assistant answers only from the guides and cites them — and says "I don't know"
when the guides are silent.

In [7]:
from farmwise.rag.assistant import ask, ask_without_rag

q = "How do I raise tilapia fish in a pond?"   # not in the farming guides
print("PLAIN LLM (no sources, may hallucinate):")
print(" ", ask_without_rag(q).answer.replace("\n", " ")[:300])

print("\nRAG (grounded in guides):")
rag = ask(q)
print(" ", rag.answer)
print("  sources:", [s.source for s in rag.sources] or "(none that answer this)")

PLAIN LLM (no sources, may hallucinate):


  Here's a simple guide to raising tilapia fish in a pond:  **Preparation**  1. **Choose a suitable pond**: Tilapia prefer warm water (75-85°F) and a pH range of 6.5-8.5. Select a pond with a minimum size of 1 acre. 2. **Test the water**: Check the water quality and pH before introducing fish. 3. **Ad

RAG (grounded in guides):


  I don't know based on the farming guides provided.
  sources: ['sample_tomato.md', 'sample_tomato.md', 'sample_maize.md']


## 7 — Limitations (say these out loud)

- **Geography:** the crop data was not collected in Ghana; some local conditions
  fall outside its training range. Treat recommendations as a starting point,
  validate with local extension officers.
- **No ground truth here:** high test accuracy is on *this* dataset's distribution,
  not on Ghanaian farms.
- **Confidence ≠ correctness:** the model is sometimes confidently wrong,
  especially out of range.
- **RAG is only as good as its guides:** sample guides here are placeholders;
  real MoFA/FAO materials must replace them, and answers can still misread a passage.
- **No memory, no users, not shareable yet** — that's what Session 2 addresses.

**Responsible takeaway:** ship the uncertainty with the answer — OOD flags,
confidence, and citations — so a human stays in the loop.